# Feature Selection

Scores every feature of the main dataset's **training** split (from
`final_data/`, so the test data never influences the selection) with two
order-agnostic methods - mutual information and RF permutation importance
(stratified 5-fold, macro-F1, class-weighted for imbalance) - prunes
redundant pairs (|Spearman rho| > 0.9), and writes the top-k list to
`configs/config.json` as `selected_features` (the only write this notebook
makes). The dataset, the file paths and `top_k` all come from the config;
when feature engineering is run for this dataset, re-running this notebook
refreshes the selection.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold

root = Path.cwd()
while not (root / "configs" / "config.json").is_file():
    if root == root.parent:
        raise FileNotFoundError("configs/config.json not found above the working directory")
    root = root.parent

CONFIG_PATH = root / "configs" / "config.json"
config = json.loads(CONFIG_PATH.read_text())
RANDOM_STATE = config["training"]["random_state"]


def dataset_file(ds_id, stage):
    """Path of one dataset file; config paths are relative to configs/."""
    ds = config["datasets"]["list"][ds_id]
    entry = ds["files"][stage]
    if isinstance(entry, dict):
        entry = entry.get("train", entry.get("data"))
    elif isinstance(entry, list):
        entry = entry[0]
    return root / "configs" / config["paths"]["datasets_dir"] / ds["folder"] / entry


def prepare(ds_id):
    """Numeric matrix + target from the final training split (in-memory)."""
    ds = config["datasets"]["list"][ds_id]
    df = pd.read_csv(dataset_file(ds_id, "final"), na_values=ds["na_values"] or None)
    target = ds["target"]["column"]
    y = df[target].map({c: i for i, c in enumerate(ds["target"]["values"])})
    X = df.drop(columns=[target])
    for c in X.columns:
        if pd.api.types.is_string_dtype(X[c]):
            X[c] = X[c].map({v: i for i, v in enumerate(sorted(X[c].dropna().unique()))})
    return X.fillna(X.median(numeric_only=True)), y

In [2]:
def mi_scores(X, y, discrete):
    s = mutual_info_classif(X, y, discrete_features=discrete, random_state=RANDOM_STATE)
    return pd.Series(s, index=X.columns, name="mutual_info")


def permutation_scores(X, y):
    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                random_state=RANDOM_STATE, n_jobs=-1)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    imp = np.zeros(X.shape[1])
    for train, test in skf.split(X, y):
        rf.fit(X.iloc[train], y.iloc[train])
        r = permutation_importance(rf, X.iloc[test], y.iloc[test],
                                   scoring="f1_macro", n_repeats=5,
                                   random_state=RANDOM_STATE, n_jobs=-1)
        imp += r.importances_mean
    return pd.Series(imp / skf.get_n_splits(), index=X.columns, name="perm_importance")

In [3]:
ds_id = config["datasets"]["main_dataset"]
top_k = config["feature_selection"]["top_k"]

X, y = prepare(ds_id)
discrete = np.array([X[c].nunique() <= 6 for c in X.columns])
mi = mi_scores(X, y, discrete)
perm = permutation_scores(X, y)

table = pd.DataFrame({"mutual_info": mi, "perm_importance": perm})
table["mi_rank"] = mi.rank(ascending=False, method="min")
table["perm_rank"] = perm.rank(ascending=False, method="min")
table["avg_rank"] = (table["mi_rank"] + table["perm_rank"]) / 2
table = table.sort_values("avg_rank")

corr = X.corr(method="spearman").abs()
kept = []
for col in table.index:
    if any(corr.loc[col, k] > 0.9 for k in kept):
        continue
    kept.append(col)
pruned = [c for c in table.index if c not in kept]

print(f"===== {ds_id} - {config['datasets']['list'][ds_id]['name']} =====")
print(f"training rows: {len(y)} | scored {X.shape[1]} columns over {y.nunique()} classes; "
      f"redundancy pruning removed: {pruned or 'nothing'}\n")
print(table.round(4).to_string())

rolled = table.loc[kept].copy()
grouped = rolled.groupby(rolled.index).agg(
    mutual_info=("mutual_info", "max"),
    perm_importance=("perm_importance", "max"),
    avg_rank=("avg_rank", "min"),
).sort_values("avg_rank")
print("\nFinal ranking:")
print(grouped.round(4).to_string())

selected = {ds_id: list(grouped.index[:top_k])}
print(f"\nPROPOSED selected_features (top {top_k} by average rank):")
print(selected[ds_id])

d:\Softwares\envs\f21dl\Lib\site-packages\sklearn\metrics\cluster\_supervised.py:68: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)


===== DS1 - UCI Chronic Kidney Disease Dataset =====
training rows: 320 | scored 30 columns over 2 classes; redundancy pruning removed: ['bu', 'log_sc', 'sc', 'urine_abnormality_score']

                         mutual_info  perm_importance  mi_rank  perm_rank  avg_rank
hemo                          0.4421           0.0115      1.0        2.0       1.5
sg                            0.3293           0.0150      6.0        1.0       3.5
bu_sc_interaction             0.2880           0.0000      9.0        3.0       6.0
htn                           0.2161           0.0000     12.0        3.0       7.5
pot                           0.2118           0.0000     13.0        3.0       8.0
dm                            0.2004           0.0000     14.0        3.0       8.5
log_bu                        0.1966           0.0000     15.0        3.0       9.0
bu                            0.1945           0.0000     16.0        3.0       9.5
bp                            0.1902           0.0000    

In [4]:
for ds_id, features in selected.items():
    config["datasets"]["list"][ds_id]["selected_features"] = features
CONFIG_PATH.write_text(json.dumps(config, indent=4, ensure_ascii=False) + "\n")
print("wrote selected_features to configs/config.json for:", ", ".join(selected))

wrote selected_features to configs/config.json for: DS1
